# Reproducing the CRC multicellular factor analysis (MOFACell)

This notebook reproduces, as closely as possible with MINA, the multicellular
factor analysis of the Hamburg colorectal-cancer (CRC) MIBI study
(`notebooks/multicellular/MOFACell.ipynb` in the study repository). It starts
from the **precomputed** feature object the study assembled,
`celltype_features_with_functional_markers.h5mu`, so the only modelling choices
that differ from the original are those of the factor-decomposition engine.

The original ran **MuVI** with `nmf=False` (a linear model, *without* enforced
non-negativity). MINA performs the decomposition with
[MOFA-FLEX](https://github.com/bioFAM/mofaflex); we use a Normal likelihood with
a spike-and-slab weight prior, which is the matching linear, non-negativity-free
setup.

!!! note "About the data"
    This example expects the precomputed
    `celltype_features_with_functional_markers.h5mu` (8 cell-type views over
    field-of-view samples, each carrying cell-type abundance, metabolic and
    functional markers, morphology, and MISTy spatial lineage-interaction
    features, plus a `Stage` covariate). Set `DATA_DIR` below. It is not shipped
    with MINA.

!!! tip "Requirements"
    `pip install "mina[spatial]"` and a working `mofaflex` install.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import mudata as md
import matplotlib.pyplot as plt

import mofaflex as mf
import mina

DATA_DIR = Path("path/to/MIBI-Analysis_Hamburg_CRC_TMA_2024/data")
features = md.read(DATA_DIR / "celltype_features_with_functional_markers.h5mu")
features

## 1. The precomputed multi-view object

Samples are fields of view (FOVs); the eight modalities are cell-type lineages.
Each view stacks the same feature blocks: cell-type **abundance**
(`proportion`), **metabolic** and **functional** markers, **morphology**, and
the **MISTy** spatial lineage-interaction features. Clinical tumour stage is in
`features.obs["Stage"]`.

In [ ]:
anndata_dict = {view: features[view].copy() for view in features.mod}

metadata = features.obs.copy()
metadata.index = metadata.index.astype(str)
metadata["sample_id"] = metadata.index

{view: ad.shape for view, ad in anndata_dict.items()}

## 2. Preprocessing

MOFACell centred and scaled every feature within each modality. MINA reproduces
this exactly with `norm_log(method="zscore")` (missing values are kept as NaN
and handled by the model). Features are then prefixed with their view.

In [ ]:
mina.up.norm_log(anndata_dict, method="zscore")
mina.up.utils.append_view_to_var(anndata_dict)

## 3. Factor decomposition (linear, non-negativity-free)

Ten factors, matching the original MuVI `n_factors=10`, `nmf=False` run. The
spike-and-slab weight prior with a Normal likelihood is the linear,
non-negativity-free MOFA-FLEX equivalent.

In [ ]:
mdata_model = md.MuData(anndata_dict)

model = mf.terms.MofaFlex(n_factors=10, weight_prior="SpikeSlab", init_factors="pca")
model.fit(
    mdata_model,
    seed=0,
    lr=0.01,
    early_stopper_patience=1000,
    likelihoods="Normal",
    subset_var=None,
    save_path=False,
)

amodel = mina.down.model_to_anndata(
    anndata_dict=anndata_dict, metadata=metadata, model=model
)
amodel

## 4. Variance explained and reconstruction

A macro-R² around 0.4 reproduces the original report.

In [ ]:
variance_df = mina.down.variance_by_view_info(amodel)
mina.pl.plot_variance_by_view(variance_df)

In [ ]:
reconstruction = mina.down.reconstruction_info(amodel)
print("Macro R2:", round(reconstruction["macro"]["R2"], 3))

## 5. Variance explained per feature class

As in the original, metabolic and functional markers carry most of the captured
variance. The spatial feature list is everything that is not abundance,
metabolism, function or morphology (i.e. the MISTy interaction features).

In [ ]:
metab = ["CA9", "CD98", "CytC", "MCT1", "ASCT2", "LDH", "GS", "GLS",
         "ATP5A", "CS", "PKM2", "GLUT1", "ARG1", "CPT1A", "Ki67"]
func = ["PD1", "PDL1", "STING1", "MSH2", "MSH6"]
morpho = ["eccentricity", "perimeter", "area"]
morpho = morpho + [f"{m}_std" for m in morpho]

all_features = set()
for view in amodel.obsm:
    all_features |= {str(c) for c in amodel.uns[f"{view}_columns"]}
spatial = sorted(all_features - {"proportion", *metab, *func, *morpho})

feature_type_map = {
    "Abundance": ["proportion"],
    "Metabolism": metab,
    "Function": func,
    "Morphology": morpho,
    "Spatial": spatial,
}
featureclass_df = mina.down.featureclass_variance_info(amodel, feature_type_map=feature_type_map)
mina.pl.plot_featureclass_variance(featureclass_df)

## 6. Which factors track tumour stage?

Kruskal-Wallis tests link factors to pT stage. As in the original several
factors are significant after correction; the exact factor indices differ from
the MuVI run because the decomposition engine differs, so we read them off
programmatically rather than hard-coding them.

In [ ]:
scores = mina.down.factor_scores_info(amodel, obs_keys=["Stage"])
scores["Stage"] = pd.Categorical(
    scores["Stage"], categories=["Colon-no.", "pT1", "pT2", "pT3", "pT4"], ordered=True
)

pT_assoc = mina.down.kruskal_info(scores, group_col="Stage")  # Bonferroni by default
pT_assoc

In [ ]:
top_factor = pT_assoc.iloc[0]["factor"]
mina.pl.plot_factor_violin(scores, factor=top_factor, group_col="Stage")

## 7. Stage separation in factor space (biplot)

Reproducing the BiplotFactors figure: the two leading stage-associated factors
with per-stage 2-SD confidence ellipses.

In [ ]:
f_x, f_y = pT_assoc.iloc[0]["factor"], pT_assoc.iloc[1]["factor"]
ellipse_df = mina.down.confidence_ellipses_info(scores, x_factor=f_x, y_factor=f_y, group_col="Stage")
mina.pl.plot_confidence_ellipses(scores, ellipse_df, x_factor=f_x, y_factor=f_y, group_col="Stage")

## 8. Top loadings of a stage-associated factor

The features (and views) driving the leading stage-associated factor —
the MINA equivalent of the per-factor loading heatmaps in the original.

In [ ]:
loadings = mina.down.variable_loadings_info(amodel)
mina.pl.plot_top_loadings_heatmap(loadings, factor=top_factor, top_n=30)

## Summary

Starting from the study's precomputed feature object, MINA reproduces the CRC
multicellular factor analysis: a ten-factor linear (non-negativity-free)
decomposition with macro-R² ≈ 0.4, dominated by metabolic and functional
variance, with several factors significantly associated with tumour stage and a
clear stage gradient in the leading factor biplot.

Because the decomposition engine differs from the original MuVI run, individual
factor numbers are not expected to match one-to-one; screen associations
programmatically (as above). The same `factor_scores_info` / `kruskal_info`
workflow extends directly to the study's other covariates such as nodal status
(pN) and microsatellite instability (MSI).